# Lab | Data Structuring and Combining Data

## Challenge 1: Combining & Cleaning Data

In this challenge, we will be working with the customer data from an insurance company, as we did in the two previous labs. The data can be found here:
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv

But this time, we got new data, which can be found in the following 2 CSV files located at the links below.

- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv

Note that you'll need to clean and format the new data.

Observation:
- One option is to first combine the three datasets and then apply the cleaning function to the new combined dataset
- Another option would be to read the clean file you saved in the previous lab, and just clean the two new files and concatenate the three clean datasets

In [53]:
# Your code goes here
import pandas as pd
import numpy as np

def clean_cols(df):
    df2=df.copy()
    df2.columns = df2.columns.map(lambda x: x.lower().replace(' ','_'))
    df2.rename(columns={'st': 'state'}, inplace=True)
    return df2

def clean_vals(df):
    df2=df.copy()
    df2.gender = df2.gender.replace({'M':'male','Male':'male','F':'female','Femal':'female'})
    df2.state = df2.state.replace({'AZ':'Arizona','Cali':'California','WA':'Washington'})
    df2.education = df2.education.replace({'Bachelors':'Bachelor'})
    df2.customer_lifetime_value = df2.customer_lifetime_value.apply(lambda x: x.replace('%','') if isinstance(x,str) else x)
    df2.vehicle_class = df2.vehicle_class.replace({'Sports Car':'Luxury','Luxury SUV':'Luxury','Luxury Car':'Luxury'})
    return df2

def convert_numeric(df):
    df2 = df.copy()
    df2.customer_lifetime_value = df2.customer_lifetime_value.replace(['', ' '], np.nan).astype(float)
    df2['number_of_open_complaints'] = df2['number_of_open_complaints'].apply(lambda x: float(x.split("/")[1]) if isinstance(x, str) 
                                                                              and '/' in x else float(x) if isinstance(x, (int, float)) 
                                                                              or (isinstance(x, str) and x.isdigit())  else np.nan)
    df2.loc[:, 'number_of_open_complaints'] = df2.loc[:, 'number_of_open_complaints'].astype(float)
    df2.number_of_open_complaints.value_counts()
    return df2

def remove_nans(df):
    df2 = df.copy()
    df2 = df2.dropna(thresh=len(df2.columns)-1)
    df2.gender = df2.gender.fillna('unknown')
    df2.customer_lifetime_value = df2.customer_lifetime_value.fillna(df2.customer_lifetime_value.mode()[0])
    return df2

def remove_dupl(df):
    df2 = df.copy()
    cols = list(df2.columns)
    df2.drop_duplicates(subset=cols[1:], keep='last', inplace=True) 
    df2 = df2.reset_index(drop=True)
    #df.to_csv('insurance_dataset.csv', index=False) #optional save to csv
    return df2

def clean_all(df):
    df2 = df.copy()
    df2 = clean_cols(df2)
    df2 = clean_vals(df2)
    df2 = convert_numeric(df2)
    df2 = remove_nans(df2)
    df2 = remove_dupl(df2)
    return df2

custdata_df1 = pd.read_csv('https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv')
custdata_df2 = pd.read_csv('https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv')
custdata_df3 = pd.read_csv('https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv')

In [55]:
custdata_df1 = clean_all(custdata_df1)
custdata_df2 = clean_all(custdata_df2)
custdata_df3 = clean_all(custdata_df3)
df4 = (pd.concat([custdata_df1, custdata_df2, custdata_df3], axis=0))
# df4.isna().sum()

# Challenge 2: Structuring Data

In this challenge, we will continue to work with customer data from an insurance company, but we will use a dataset with more columns, called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by performing data cleaning, formatting, and structuring.

In [75]:
challenge2_df = pd.read_csv('https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv')
challenge2_df.nunique()


unnamed:_0                       10910
customer                          9134
state                                5
customer_lifetime_value           8041
response                             2
coverage                             3
education                            5
effective_to_date                   59
employmentstatus                     5
gender                               2
income                            5694
location_code                        3
marital_status                       3
monthly_premium_auto               202
months_since_last_claim             37
months_since_policy_inception      100
number_of_open_complaints            7
number_of_policies                   9
policy_type                          3
policy                               9
renew_offer_type                     4
sales_channel                        4
total_claim_amount                5106
vehicle_class                        6
vehicle_size                         3
vehicle_type             

In [78]:
challenge2_df.drop_duplicates(subset='customer', inplace=True)

In [87]:
#Using pivot, create a summary table showing the total revenue for each sales channel (branch, call center, web, and mail)
pvt = challenge2_df.pivot_table( columns='sales_channel', values=['customer_lifetime_value'], aggfunc='sum').round(2)
display(pvt)
display(pvt.head().style.format("{:,.2f}") ) #readability
#conclusion: revenue is highest from agents, then branch and call centre. Web performs very poorly.

sales_channel,Agent,Branch,Call Center,Web
customer_lifetime_value,27668955.42,20843300.35,14296651.35,10308219.18


sales_channel,Agent,Branch,Call Center,Web
customer_lifetime_value,"27,668,955.42","20,843,300.35","14,296,651.35","10,308,219.18"


In [93]:
challenge2_df.pivot_table(index='gender', columns='education', values=['customer_lifetime_value'], aggfunc='mean').style.format("{:,.2f}")
#there is an inverse correlation between customer_lifetime_value and education level. Highly educated customers, i.e. those with doctorates 
#have the lowest value, whereas those with high school or below have the highest for women, and second higest for men.

1. You work at the marketing department and you want to know which sales channel brought the most sales in terms of total revenue. Using pivot, create a summary table showing the total revenue for each sales channel (branch, call center, web, and mail).
Round the total revenue to 2 decimal points.  Analyze the resulting table to draw insights.

2. Create a pivot table that shows the average customer lifetime value per gender and education level. Analyze the resulting table to draw insights.

## Bonus

You work at the customer service department and you want to know which months had the highest number of complaints by policy type category. Create a summary table showing the number of complaints by policy type and month.
Show it in a long format table.

*In data analysis, a long format table is a way of structuring data in which each observation or measurement is stored in a separate row of the table. The key characteristic of a long format table is that each column represents a single variable, and each row represents a single observation of that variable.*

*More information about long and wide format tables here: https://www.statology.org/long-vs-wide-data/*

In [97]:
# Your code goes here
challenge2_df.pivot_table(index='month', columns='policy_type', values=['number_of_open_complaints'], aggfunc='sum').stack()


C:\Users\clair\AppData\Local\Temp\ipykernel_11504\2880740709.py:2: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  challenge2_df.pivot_table(index='month', columns='policy_type', values=['number_of_open_complaints'], aggfunc='sum').stack()


number_of_open_complaints
month policy_type                              
1     Corporate Auto                 372.296195
      Personal Auto                 1464.342220
      Special Auto                    76.768512
2     Corporate Auto                 340.527683
      Personal Auto                 1186.194123
      Special Auto                    71.152768